# OpenEnv SME Negotiator — GRPO Training on Colab

**Colab workflow** for a tiny GRPO run on the SME Liquidity environment.

This notebook is intentionally structured like the kube-sre-gym Colab flow:
Install → Config → Smoke test → Imports/utilities → GRPO setup → Train → Reward curves → Optional upload.

## 1. Install Dependencies

Install project + RL extras and common training dependencies.

In [24]:
%pip uninstall -y trl
%pip install -q "trl==0.29.0" "transformers>=4.45.0" "peft>=0.19.1" "datasets" "huggingface_hub>=0.20.0" "matplotlib" "jmespath"

Found existing installation: trl 0.29.0
Uninstalling trl-0.29.0:
  Successfully uninstalled trl-0.29.0


In [3]:
! git clone https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git
%cd OpenEnv_SME_Negotiator-2.o

%pip install -q -U pip
%pip install -q -e ".[rl]"
%pip install -q "trl>=0.29.0" "peft>=0.19.1" "transformers" "datasets" "huggingface_hub>=0.20.0" "matplotlib"

Cloning into 'OpenEnv_SME_Negotiator-2.o'...
remote: Enumerating objects: 516, done.
remote: Counting objects: 100% (516/516), done.
remote: Compressing objects: 100% (257/257), done.
remote: Total 516 (delta 259), reused 498 (delta 244), pack-reused 0 (from 0)
Receiving objects: 100% (516/516), 1.63 MiB | 20.87 MiB/s, done.
Resolving deltas: 100% (259/259), done.
/content/OpenEnv_SME_Negotiator-2.o
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.1 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for openenv-sme-negotiator (pyproject.toml) ... done


## 2. Configuration

Set model, task, and tiny training hyperparameters. Add `HF_TOKEN` in Colab Secrets (key icon) if required.

In [5]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")

# --- HuggingFace token from Colab secrets (optional) ---
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

# --- Core config ---
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TASK_NAME = "liquidity-correlation-hard"
TOTAL_PERIODS = 2

# Tiny run defaults for Colab demo
STEPS = 10
NUM_SAMPLES = 8
OUTPUT_DIR = Path("outputs/colab_grpo_sme_liquidity")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print("  MODEL_NAME   :", MODEL_NAME)
print("  TASK_NAME    :", TASK_NAME)
print("  TOTAL_PERIODS:", TOTAL_PERIODS)
print("  STEPS        :", STEPS)
print("  NUM_SAMPLES  :", NUM_SAMPLES)
print("  OUTPUT_DIR   :", OUTPUT_DIR)

Configuration:
  MODEL_NAME   : Qwen/Qwen2.5-1.5B-Instruct
  TASK_NAME    : liquidity-correlation-hard
  TOTAL_PERIODS: 2
  STEPS        : 10
  NUM_SAMPLES  : 8
  OUTPUT_DIR   : outputs/colab_grpo_sme_liquidity


## 3. Smoke Test — Verify Environment Connectivity

Run a reset on the in-process SME liquidity environment.

In [6]:
from server.environment import SMELiquidityEnvironment

env = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
obs = env.reset(seed=42, difficulty="hard", task_name=TASK_NAME)
print("Smoke test OK. Initial observation:")
print(obs)

Smoke test OK. Initial observation:
done=False reward=0.0 metadata={'episode_id': 'deal_p0_buyer_0_0-42', 'seed': 42, 'base_concede': 0.005836560790747302, 'buyer_day_floor': 30, 'task_name': 'liquidity-correlation-hard', 'task_description': 'Stage 2 correlated-risk hard mode: both buyers are slow and risky, the SME has very thin buffers, and the financier cannot absorb every invoice.', 'context_note': 'Correlated buyer risk, tight supplier terms, and limited financing force solvency and NPV trade-offs.', 'reward_mode': 'stage3_long_horizon', 'active_deal_id': 'deal_p0_buyer_0_0', 'current_period': 0, 'episode_step': 0, 'episode_step_cap': 90, 'legacy_inner_reward': None, 'legacy_reward_branch': None, 'latest_shaping_reward': 0.0, 'latest_verifiable_reward': None, 'simulation_projection_present': False, 'resolved_deal_count': 0, 'defaulted_sme_count': 0, 'last_tool_name': None, 'tool_backend_mode': 'deterministic', 'tool_bonus_applied': 0.0, 'pending_tool_bonus': 0.0, 'tool_spam_flag':

## 4. Import Training Utilities from Package

Import helper functions used by your RL demo path.

In [7]:
import matplotlib.pyplot as plt
from rl.demo import run_heuristic_episode, demo_train_grpo, run_policy_episode

print("Imports successful.")

Imports successful.


## 5. GRPO Training Setup

Run baseline heuristic episodes before training.

In [8]:
baseline_runs = [
    run_heuristic_episode(seed=i, total_periods=TOTAL_PERIODS, task_name=TASK_NAME)
    for i in range(5)
]
baseline_rewards = [item["total_reward"] for item in baseline_runs]

print("Baseline rewards:", baseline_rewards)
print("\nSample baseline transcript:\n")
print(baseline_runs[0].get("transcript", "(no transcript field)"))

Baseline rewards: [0.0, 0.0, 0.0, 0.0, 0.0]

Sample baseline transcript:

RESET :: Role=SME
EnvMessage=Real-world context (Razorpay Fix My Itch, B2B Services, itch score 82.8): Small suppliers often receive buyer payment terms of 60–90+ days while paying their own suppliers in ~30 days, widening working-capital gaps and forcing costly short-term financing (often ~18–24% APR class). Large buyers may resist shorter cycles. This environment simulates multi-round negotiation; terminal scores are computed by deterministic rules in sme_negotiator_env/graders.py. Task: Stage 2 correlated-risk hard mode: both buyers are slow and risky, the SME has very thin buffers, and the financier cannot absorb every invoice. Correlated buyer risk, tight supplier terms, and limited financing force solvency and NPV trade-offs. | Episode reset @ 2026-04-25T08:28:26.911448+00:00 (task_id=liquidity-correlation-hard, base_concede=0.0071)
Round=0 | Task=liquidity-correlation-hard | BuyerPrice=96.0 | BuyerDays=95 

## 6. Train!

Launch tiny GRPO training demo.

In [ ]:
import importlib
import inspect
from pathlib import Path

import torch
import rl.demo as demo_mod
importlib.reload(demo_mod)

def demo_train_grpo_compat(
    *,
    model_name=demo_mod.DEFAULT_MODEL_NAME,
    steps=10,
    total_periods=2,
    task_name="liquidity-correlation-hard",
    difficulty="hard",
    num_samples=8,
    seed_base=1000,
    output_dir="outputs/grpo_sme_liquidity_demo",
    use_lora=True,
    max_completion_length=256,
):
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
    from trl import GRPOConfig, GRPOTrainer

    class HistoryCallback(TrainerCallback):
        def __init__(self):
            self.steps = []
            self.avg_reward = []

        def on_log(self, args, state, control, logs=None, **kwargs):
            reward_key = None
            if logs and "episode/avg_total_reward" in logs:
                reward_key = "episode/avg_total_reward"
            elif logs and "episode/avg_base_rl_reward" in logs:
                reward_key = "episode/avg_base_rl_reward"
            if logs and reward_key is not None:
                self.steps.append(int(getattr(state, "global_step", 0) or 0))
                self.avg_reward.append(float(logs[reward_key]))
            return control

    rows = demo_mod.build_training_rows(
        prompt=demo_mod.DEFAULT_PROMPT,
        task_name=task_name,
        difficulty=difficulty,
        total_periods=total_periods,
        num_samples=num_samples,
        seed_base=seed_base,
    )
    dataset = demo_mod.build_dataset(rows)
    wrapper_cls = demo_mod.make_environment_factory(
        task_name=task_name,
        difficulty=difficulty,
        total_periods=total_periods,
        seed=seed_base,
        prompt=demo_mod.DEFAULT_PROMPT,
    )
    summary_buffer = demo_mod.EpisodeSummaryBuffer()
    reward_func = demo_mod.make_reward_function(summary_buffer=summary_buffer)
    output_path = Path(output_dir)

    cpu_only = not torch.cuda.is_available()
    if cpu_only:
        print("[compat] CUDA not available; forcing CPU-safe GRPO settings.")

    torch_dtype = None
    if torch.cuda.is_available():
        torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    tokenizer = demo_mod.configure_tokenizer(AutoTokenizer.from_pretrained(model_name))
    # Newer TRL expects response parsing metadata for tool-use flows.
    if not getattr(tokenizer, "response_schema", None):
        try:
            from trl.chat_template_utils import qwen3_schema
            tokenizer.response_schema = qwen3_schema
        except (ImportError, AttributeError):
            tokenizer.response_schema = {
                "x-regex": r"^(?P<content>.*?)(?:<\|im_end\|>|$)",
                "type": "object",
                "properties": {
                    "role": {"const": "assistant"},
                    "content": {"type": "string"},
                },
            }
        print("[compat] Applied fallback tokenizer.response_schema.")

    model_kwargs = {"low_cpu_mem_usage": True}
    if torch_dtype is not None:
        model_kwargs["torch_dtype"] = torch_dtype
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    except TypeError:
        model_kwargs.pop("low_cpu_mem_usage", None)
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    if hasattr(model, "config"):
        model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    if use_lora:
        try:
            from peft import LoraConfig, get_peft_model
            lora_config = LoraConfig(
                r=8,
                lora_alpha=16,
                lora_dropout=0.05,
                bias="none",
                task_type="CAUSAL_LM",
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            )
            model = get_peft_model(model, lora_config)
            if hasattr(model, "print_trainable_parameters"):
                model.print_trainable_parameters()
        except ImportError:
            print("[compat] peft is not installed; falling back to full-parameter training.")

    metrics_callback = demo_mod.build_metrics_callback(
        summary_buffer,
        TrainerCallback,
        curriculum=None,
        build_preference_dataset=False,
        scorer=None,
        output_dir=str(output_path),
    )
    history_callback = HistoryCallback()

    grpo_kwargs = {
        "output_dir": str(output_path),
        "remove_unused_columns": False,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 2,
        "num_generations": 2,
        "generation_batch_size": 2,
        "learning_rate": 5e-6,
        "max_prompt_length": 512,
        "max_completion_length": max(64, int(max_completion_length)),
        "max_steps": max(1, int(steps)),
        "logging_steps": 1,
        "save_steps": max(1, int(steps)),
        "report_to": "none",
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "bf16": torch_dtype == torch.bfloat16,
        "fp16": torch_dtype == torch.float16,
        "use_cpu": cpu_only,
    }
    valid_keys = set(inspect.signature(GRPOConfig.__init__).parameters) - {"self"}
    dropped = sorted(k for k in grpo_kwargs if k not in valid_keys)
    if dropped:
        print("[compat] Dropping unsupported GRPOConfig args:", dropped)
    training_args = GRPOConfig(**{k: v for k, v in grpo_kwargs.items() if k in valid_keys})

    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=reward_func,
        train_dataset=dataset,
        args=training_args,
        environment_factory=wrapper_cls,
        callbacks=[metrics_callback, history_callback],
    )
    trainer.train()

    checkpoint_path = output_path / "final-demo-model"
    trainer.save_model(str(checkpoint_path))
    if hasattr(tokenizer, "save_pretrained"):
        tokenizer.save_pretrained(str(checkpoint_path))

    return {
        "steps": history_callback.steps,
        "avg_reward": history_callback.avg_reward,
        "output_dir": str(output_path),
        "checkpoint_path": str(checkpoint_path),
    }

demo_mod.demo_train_grpo = demo_train_grpo_compat
demo_train_grpo = demo_mod.demo_train_grpo
print("demo_train_grpo compatibility patch ready.")

demo_train_grpo compatibility patch ready.


In [25]:
print("Starting GRPO training...")
print(f"  Model      : {MODEL_NAME}")
print(f"  Task       : {TASK_NAME}")
print(f"  Steps      : {STEPS}")
print(f"  Samples    : {NUM_SAMPLES}")
print(f"  Output dir : {OUTPUT_DIR}")

history = demo_train_grpo(
    model_name=MODEL_NAME,
    steps=STEPS,
    total_periods=TOTAL_PERIODS,
    task_name=TASK_NAME,
    num_samples=NUM_SAMPLES,
    output_dir=str(OUTPUT_DIR),
)

print("Training done.")
history

Starting GRPO training...
  Model      : Qwen/Qwen2.5-1.5B-Instruct
  Task       : liquidity-correlation-hard
  Steps      : 10
  Samples    : 8
  Output dir : outputs/colab_grpo_sme_liquidity
[compat] CUDA not available; forcing CPU-safe GRPO settings.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[compat] Dropping unsupported GRPOConfig args: ['max_prompt_length']


/tmp/ipykernel_2626/1197101748.py:99: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(


ValueError: Unrecognized chat template, failed to add response schema. Please manually set the response schema on the tokenizer or processor. See the Transformers [docs](https://huggingface.co/docs/transformers/main/en/chat_response_parsing#response-parsing) for more details on response parsing.

## 7. Reward Curves

Visualize average reward across training steps.

In [ ]:
steps = history.get("steps", [])
avg_reward = history.get("avg_reward", [])

plt.figure(figsize=(7, 4))
plt.plot(steps, avg_reward, marker="o")
plt.xlabel("Training step")
plt.ylabel("Average episode reward")
plt.title("Tiny GRPO Demo on SMELiquidityEnvironment")
plt.grid(alpha=0.3)
plt.show()

## 8. Push to Hub (Optional)

Optional: evaluate before/after and keep checkpoint path for later upload scripts.

In [ ]:
from rl.demo import run_policy_episode

print("=== Before Training (Heuristic) ===")
print(run_policy_episode(policy="heuristic", seed=123, total_periods=TOTAL_PERIODS, task_name=TASK_NAME))

print("\n=== After Tiny GRPO Demo ===")
print(
    run_policy_episode(
        policy="trained",
        seed=123,
        total_periods=TOTAL_PERIODS,
        task_name=TASK_NAME,
        checkpoint_path=history.get("checkpoint_path"),
    )
)

print("\nCheckpoint path:", history.get("checkpoint_path"))
# If you have a push helper in your repo, call it here.

## 9. Publish Model Card and Checkpoint to Hugging Face

Set `HF_TOKEN` in Colab Secrets, then update `HF_REPO_ID` and run this cell after training succeeds.

In [ ]:
from huggingface_hub import HfApi

HF_REPO_ID = "YOUR_USERNAME/openenv-sme-negotiator-grpo"
MODEL_DIR = Path(history.get("checkpoint_path", OUTPUT_DIR / "final-demo-model"))
CARD_PATH = Path("huggingface/model_card.md")

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Checkpoint folder not found: {MODEL_DIR}")
if not CARD_PATH.exists():
    raise FileNotFoundError(f"Model card not found: {CARD_PATH}")
if HF_REPO_ID.startswith("YOUR_USERNAME/"):
    raise ValueError("Set HF_REPO_ID to your Hugging Face repo id before publishing.")

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(MODEL_DIR),
    ignore_patterns=["*.tmp", "*.log", "checkpoint-*"],
)
api.upload_file(
    repo_id=HF_REPO_ID,
    repo_type="model",
    path_or_fileobj=str(CARD_PATH),
    path_in_repo="README.md",
)
print(f"Published: https://huggingface.co/{HF_REPO_ID}")